# Agilent BioTek 405 TS plate washer quickstart

The 405 TS is a microplate washer. Everything it does goes through a single wash manifold: it
primes its fluid lines, dispenses buffer into wells, aspirates them empty, and washes a plate by
repeating the two. It carries no syringes and no peristaltic pumps, so it dispenses nothing of its
own beyond what the manifold delivers.

This quickstart connects to the washer, reads what it is and what it has fitted, tells it which
plate is on its carrier, primes the lines, runs a small batch, explains where in the well a step
works, runs a protocol file, and disconnects.

| Property | Value |
|---|---|
| Communication | A serial port, or USB through an FTDI interface |
| Serial parameters | 38400 baud, 8 data bits, 2 stop bits, no parity, no flow control |
| Operations | Priming, dispensing, aspirating, washing, auto-clean, shake and soak |
| Plate formats | 96-well, 384-well and 384-well PCR, resolved from the PyLabRobot plate resource |
| Buffer inlets | A, B, C, D |
| Manifold reach | Depth 1-255 motor steps, across the well -60 to 60, along the well -60 to 60 |
| Protocol files | `.LHC` protocol files are read, checked and run |

```{warning}
Follow the manufacturer's installation, fluid-handling and safety instructions. A prime and a
dispense both move fluid: the buffer bottle must be full and the waste bottle empty enough before
anything in this notebook runs.
```

```{device-card} biotek-405-ts
```

## How it communicates

PyLabRobot frames each command as an 11-byte header and a payload, writes it to the instrument,
reads back an acknowledgement and the reply, and turns a non-zero status into a typed exception.
Which of the two transports carries those bytes is decided by the port string alone, and nothing
above that point knows which one it got.

Operations that move fluid do not answer when they are done. The driver sends them, then polls the
instrument's run state until it stops reporting a step in progress, which is why every method that
touches the instrument is awaited and can take as long as the physical operation does.

Install PyLabRobot with its serial dependencies, or with the FTDI ones if the washer is on USB.

Reading `.LHC` protocol files additionally needs `pycryptodome`, which is not a PyLabRobot
dependency: the file format is encrypted, and the cipher is not in the standard library. Leave it
out if you only build protocols in Python — `read()` raises a `RuntimeError` telling you to install
it if you later try to read a file without it.

In [ ]:
# %pip install "pylabrobot[serial]" pycryptodome

# On USB, install the FTDI dependencies instead: "pylabrobot[ftdi]".

# validation mode also drives an emulated serial port, so it needs the serial extra whichever
# transport the instrument itself is on.
# %pip install "pylabrobot[serial,ftdi]" pycryptodome

# development: install this fork from the local checkout instead of PyPI.
%pip install -e "/home/stefan/workspace/biotek_plr/pylabrobot_modified[serial,ftdi]" pycryptodome


## Physical setup and finding the port

Install, plumb and power the washer according to the manufacturer's instructions. Connect the
buffer bottles to the inlets the protocol names, connect the waste bottle, and connect the
instrument to the computer.

Then find the port string:

- **Serial.** Pass the operating system's own name for the port: `COM3` on Windows,
  `/dev/ttyUSB0` or `/dev/ttyS4` on Linux and macOS. Anything that is not a USB serial number is
  taken to be a serial port and passed through unexamined, so no particular naming pattern is
  required.

- **USB.** List the attached FTDI devices and use the reported serial number:

  ```bash
  python -m pylibftdi.examples.list_devices
  ```

  The port is then `ftdi:<serial>`, for example `ftdi:183193P`. The form the instrument's own
  protocol files record, `USB 405 TS sn:183193P`, is accepted as well.

Keep the carrier and the area around it clear from here on. Nothing in this notebook moves the
carrier before the batch section, but a fault can home the motors at any time.

## Turn on logging

The driver reports what it is doing through the standard library's `logging`, and says nothing
otherwise. Without this cell every step the instrument runs passes silently.

In [ ]:
import logging

# logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

# validation mode: DEBUG is what makes the driver log each frame it sends and receives. The
# structured record of both sides is the journal below; this is the running commentary.
logging.basicConfig(level=logging.DEBUG, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("pylabrobot.agilent.biotek.lhc").setLevel(logging.DEBUG)

## See which ports have something behind them

`pyserial` lists every serial port the operating system offers, most of which are kernel
placeholders with no hardware behind them — on Linux the 32 `/dev/ttyS*` entries, which report
their description and hardware id as `n/a`. Skipping those leaves the ports worth trying, and a
USB-serial adapter names the instrument it is wired to, so the washer is usually recognisable at a
glance.

This lists serial ports only. A washer driven through the FTDI transport is found with
`python -m pylibftdi.examples.list_devices` instead — though a device the kernel has bound to its
own FTDI serial driver shows up here too, and can be used either way.

In [ ]:
from serial.tools.list_ports import comports

candidates = [port for port in comports() if port.description != "n/a" or port.vid is not None]

for port in candidates:
    serial_number = f"  sn:{port.serial_number}" if port.serial_number else ""
    print(f"{port.device:16} {port.description}{serial_number}")

if not candidates:
    print("no port has a device behind it; is the washer powered and connected?")

## Turn on validation mode

Validation mode runs the vendor's own driver alongside this one. Every operation the washer is
asked to carry out is put to the real interface DLL as well — loaded under wine, talking to an
emulated serial port, and answered with the replies this instrument actually gave — and the two
are compared frame for frame. Everything both sides send and receive goes to `shadow.jsonl`.

```{note}
This is a development harness, not part of PyLabRobot. It lives in `verification/plr_shadow/`
next to the trial server and needs that setup: wine, a `tty0tty` pair per product, and the DLL
build for this model populated under `trial_server/interface_host/products/`. To run this
notebook as a user would, skip this cell and the *validation mode* lines in the cells below —
each one leaves the ordinary command commented out directly above it.
```

Attaching the mirror does not change what the driver does. It is an observer: it is told each
operation and each frame after the fact, its failures are logged rather than raised, and a watched
run puts exactly the same bytes on the wire as an unwatched one.

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

# Both relative to this notebook's own directory. Point them at your checkout if you start
# Jupyter from elsewhere.
PACKAGE = Path("../../../..").resolve()  # the pylabrobot checkout this notebook lives in
WORKSPACE = PACKAGE.parent

# The harness has to import the same checkout this notebook does. Without this it would pick up
# whatever `pylabrobot` happens to be installed, and compare against a different driver than the
# one being run.
sys.path.insert(0, str(PACKAGE))
sys.path.insert(0, str(WORKSPACE / "verification"))
sys.path.insert(0, str(WORKSPACE / "lhc_python_tranl" / "trial_server"))

from plr_shadow.journal import Journal
from plr_shadow.mirror import Context, Mirror
from pylabrobot.agilent.biotek.lhc.devices.instrument_settings import InstrumentSettings
from simulation_server import SimulationPool

JOURNAL_PATH = Path("shadow.jsonl")


def shadow_context() -> Context:
    """What the DLL has to be told before it is asked to do the same thing.

    The family comes from the class rather than from `device.settings`: which model this is was
    declared when the device was built, while the record only holds it once `setup()` has read the
    instrument -- and the first operations of a run finish before that.

    For the same reason the record is read through the field rather than through `device.settings`,
    which refuses to answer before the instrument has been asked. The operations that finish before
    it has -- the liveness check and the fitted-options read itself -- are mirrored against a
    plainly equipped instrument of the right model, which is the most that can be said then.
    """
    family = type(device).family
    reported = device._runtime.reported_settings
    return Context(
        instrument=int(family),
        settings=replace(reported, family=family)
        if reported is not None
        else InstrumentSettings(family=family),
        plate_type=device.plate.plate_type if device.plate is not None else None,
    )


pool = SimulationPool()
journal = Journal(JOURNAL_PATH)
mirror = Mirror(pool=pool, context=shadow_context, journal=journal)

## Build the washer

Constructing the object opens nothing and touches no hardware. It records which port to use, which
model this is, and what to call the instrument in logs and error messages.

Replace the port with the one found above.

In [ ]:
from pylabrobot.agilent.biotek.lhc import Washer405TS

# The port is a serial one unless it carries a device serial number: on USB, pass
# "ftdi:YOUR_SERIAL" instead.
# device = Washer405TS(port="COM3", name="405 TS")

# validation mode: the observer is the only change to how the device is built, and it cannot
# change what the device does -- it is told what happened, after it happened.
device = Washer405TS(port="/dev/ttyUSB1", name="405 TS", observer=mirror)
device

## Connect

`setup()` opens the link, asks whether anything is listening, and reads the options the instrument
has fitted. That read is not optional: every step is encoded against it and every check measures
against it, so a failure here stops the notebook rather than being carried past.

It raises a `BiotekError` if the port will not open, if nothing answers on it, or if the fitted
options cannot be read.

In [ ]:
await device.setup()

## Ask what answered

**The model is declared, not discovered.** The instrument does not report which model it is, so it
is the class you constructed that decides how every step is encoded and which options are read.
Pointing `Washer405TS` at a different model does not raise: it connects, and then encodes steps for
the wrong machine. Note that `device.settings.family` is not a check on this — it echoes what was
declared, not what is attached.

What the instrument does report is its serial number, which identifies the individual instrument,
and its firmware version record, whose part number says which instrument the installed firmware
image is built for.

In [ ]:
print("serial number:  ", await device.get_serial_number())

version = await device.get_firmware_version()
print("part number:    ", version.part_number)
print("firmware:       ", version.software_version)
print("data version:   ", version.data_version)

## Read the configuration

What `setup()` read is kept as a read-only record of what somebody fitted to this instrument. It is
read-only because the instrument is: of its whole command vocabulary, almost nothing about the
configuration can be written, so a record that could be edited would only mislead.

It is worth looking at, because it decides what the checks below allow. The cell washing module
unlocks the slowest travel and flow rates; buffer switching decides which inlets can be named; the
Y axis decides whether the manifold can be offset along the well at all.

In [ ]:
settings = device.settings

print("family:            ", settings.family.name)
print("wash manifold:     ", settings.washer_manifold.name)
print("valve box:         ", settings.valve_box.name)
print("buffer switching:  ", settings.buffer_switching)
print("vacuum filtration: ", settings.vacuum_filtration)
print("cell washing:      ", settings.cell_washing)
print("ultrasonic:        ", settings.ultrasonic)
print("Y axis:            ", settings.y_axis_installed)

## Ask what it can run

A model can be built to run a fixed set of operations, and a particular instrument runs the subset
its fitted hardware supports. This is that subset, and it is the answer to "why was my step
refused" before you have written the step.

In [ ]:
for step_type in device.get_available_steps():
    print(step_type.name)

## Tell it which plate is on the carrier

Nothing runs until a plate has been set. Every step carries the height it works at, measured from
the nominal heights of the format on the carrier, so without one there is nothing to measure from
and the driver raises `RejectedError` rather than guessing.

The format is resolved from the PyLabRobot plate resource itself — its columns, its rows, and how
deep its wells are. Labware that does not land on exactly one of the formats this model works is an
error naming the candidates, never a nearest fit. A 405 TS works three: 96-well, 384-well and
384-well PCR.

Which 384-well labware suits the instrument depends on the manifold fitted to it, because the
smallest volume the manifold dispenses does. A 128-tube or a 96-tube single-action manifold is
refused on a 384-well plate outright; a 192-tube manifold dispenses from 25 µL per well, and a
96-tube dual-action one from 50 µL. Read `settings.washer_manifold` above, and pick a plate whose
wells hold what your manifold will dispense — the plate below takes about 190 µL.

In [ ]:
from pylabrobot.resources import Plate_384_Well

plate = Plate_384_Well(name="plate")
device.set_plate(plate)

print(device.plate)

Some formats are never resolved from a resource, because they share their column and row count
with an ordinary plate and differ in something the resource does not carry — a well shape, a
flange, a tube, or that it is calibration labware. On a 405 TS that is the 384-well PCR format,
which is worked from a different height than a flat 384-well plate. Name one of those outright,
and use the same argument when a plate is to be worked as something other than what it resolves
to.

In [ ]:
from pylabrobot.agilent.biotek.lhc.enums.plates.plate_type import PlateType

device.set_plate(plate, plate_type=PlateType.PLATE_384_WELL_PCR)
print(device.plate)

device.set_plate(plate)  # back to what it resolves to on its own

## Check before running

`can_run()` measures steps against the instrument as it is now — what is fitted, which plate is on
the carrier, and what the plate will accept — and touches nothing. It is what `run_protocol()` does
first, so calling it yourself is how you see a refusal without moving anything.

The report is truthy when everything can run, and prints as the list of what cannot.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.manifold_prime import ManifoldPrime

report = await device.can_run([ManifoldPrime(volume=10_000, buffer="A", flow_rate=9)])
print(report)
print("can run:", bool(report))

A step this washer has no hardware for is refused with the reason, rather than failing partway
through. A syringe dispense is the clearest case: there are no syringes on a 405 TS.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.syringe_dispense import SyringeDispense

print(await device.can_run([SyringeDispense(volume=50)]))

## Prime the wash manifold

Priming pumps buffer through the manifold until its lines are full and sends it to waste. Nothing
is dispensed into the plate, which makes it the safest operation to try first — but it does move
fluid, so check the buffer and waste bottles before running this cell.

The volume is in microlitres; the instrument meters it at millilitre resolution. The default is
40 mL, which is a full prime of dry lines; 10 mL is enough to see the pump run.

The call returns when the instrument reports the step finished, which takes as long as the pumping
does.

In [ ]:
await device.washer.prime(volume=10_000, buffer="A", flow_rate=9)

## Run several operations in one batch

Opening a batch homes the motors, takes the instrument so that nothing else can interleave a run on
it, and holds it until the block ends. Every operation opens one; doing it once around several
operations is what stops the motors being homed between each of them.

The block below primes the lines and then dispenses 50 µL of buffer A into every well. **This one
dispenses into the plate**, so put a plate on the carrier that you are willing to fill.

50 µL is the smallest volume a 96-tube dual-action manifold dispenses, and a 192-tube one goes down
to 25 µL. Above that the volume is yours: nothing in the driver measures it against the well, since
the check accepts anything from the manifold's floor up to 3000 µL. A refusal here reads
`Washer Dispense Volume must be 50..3000`.

`home_on_close=True` drives the transport home before the batch closes. The instrument does not do
this by itself; ask for it when the next thing to touch the plate is a person.

Nesting is allowed and does nothing: an operation called inside an open batch joins it rather than
opening a second one.

In [ ]:
async with device.batch(home_on_close=True):
    await device.washer.prime(volume=10_000, buffer="A")
    await device.washer.dispense(volume=50, buffer="A", flow_rate=7)

A wash is the two of them repeated, and is one step rather than a loop: the instrument runs the
cycles itself. This aspirates each well empty and refills it, three times.

The refill needs a volume of its own. The dispense a wash owns is checked like any other, and
the step type defaults to no volume at all, so a wash that names none is refused before
anything moves.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.step_parts.positioning import Positioning
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.manifold_dispense import ManifoldDispense

await device.washer.wash(
    cycles=2,
    dispense=ManifoldDispense(
        volume=50,
        buffer="A",
        positioning=Positioning(z_steps=device.plate.manifold_dispense_height),
    ),
)

## Where in the well a step works

Every operation that reaches into the plate takes a `positioning`, and defaults it to the nominal
position for that head over the format on the carrier. Three numbers:

- `z_steps` — how deep the manifold goes. **This is a height, not an offset**: it defaults to the
  plate record's own nominal height for the head, and giving a larger number reaches further down
  into the well. On a 405 TS it must be 1-255.
- `x_steps` — across the well, -60 to 60 for the wash manifold.
- `y_steps` — along the well, -60 to 60 on a 405 TS. Needs the Y axis to be fitted.

The nominal heights come from the plate record, so they change with the plate and differ per head:
dispensing sits higher than aspirating, which has to reach the bottom of the well.

In [ ]:
print("nominal manifold dispense height:", device.plate.manifold_dispense_height)
print("nominal manifold aspirate height:", device.plate.manifold_aspirate_height)

To dispense a little higher than nominal — down the side of the well rather than into the middle
of it — build a `Positioning` from the nominal height rather than from a number you have written
down, so the same code stays right when the plate changes.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.step_parts.positioning import Positioning

await device.washer.dispense(
    volume=50,
    buffer="A",
    positioning=Positioning(
        z_steps=device.plate.manifold_dispense_height - 10,  # 10 steps higher in the well
        x_steps=20,  # toward one side
        y_steps=0,
    ),
)

### Why motor steps and not millimetres

PyLabRobot's convention is millimetres, and this is the one place the package deviates from it. The
field names say so outright — `z_steps`, not `z` — because the conversion is not a single number:
it differs per axis, per model, and per head, and only part of it is established.

What is known: the across-the-plate axis is **0.04572 mm per motor step**, confirmed twice over
against a published maximum offset. The depth axis has at least two scales, chosen by a property
that follows the head, and which head takes which is not settled. The along-the-plate axis is not
established at all.

Converting on the strength of that would put a manifold at the wrong depth on some head of some
model, so the package does not convert. If you need millimetres on your instrument, measure them:
drive a known offset on each axis and see where the manifold goes. A step is also what a protocol
file stores, which is what lets a protocol be read, checked and written with no instrument to
ask.

## Watch a step, and stop it

`get_status()` reports what the instrument is doing, which timed phase a running step is in, and
how many seconds are left in it. It can be called at any time, including while a step is running.

An operation does not return until its step has finished, so pausing or aborting means asking from
somewhere else while it runs. In a notebook that is a task.

In [ ]:
import asyncio

running = asyncio.create_task(device.washer.prime(volume=40_000, buffer="A"))
await asyncio.sleep(2)

status = await device.get_status()
print("state:    ", status.state.name)
print("activity: ", status.activity.name)
print("remaining:", status.remaining, "s")

Pause holds the step where it is; resume carries on from there.

In [ ]:
from pylabrobot.agilent.biotek.lhc.enums.status.run_state import RunState

status = await device.get_status()
if status.state is not RunState.BUSY:
    print(
        f"the device is {status.state.name}, not running a step -- start the cell above again "
        "and run this one while its step is still going"
    )
else:
    await device.pause()
    await asyncio.sleep(2)
    print((await device.get_status()).state.name)

    await device.resume()
    await running

Abort stops the running step instead. The operation that was waiting for it raises
`AbortedError`, which is a `BiotekError`, so a protocol run ends where it was stopped rather than
carrying on to the next step.

In [ ]:
import asyncio

from pylabrobot.agilent.biotek.lhc.comm.observer import Operation
from pylabrobot.agilent.biotek.lhc.enums.status.run_state import RunState
from pylabrobot.agilent.biotek.lhc.error_handling import AbortedError, BiotekError
from pylabrobot.agilent.biotek.lhc.protocols.steps.step_parts.positioning import Positioning
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.manifold_dispense import ManifoldDispense
from pylabrobot.agilent.biotek.lhc.serialization.commands.run_control import AbortStep

# from pylabrobot.agilent.biotek.lhc.error_handling import AbortedError, BiotekError

running = asyncio.create_task(device.washer.prime(volume=40_000, buffer="A"))
await asyncio.sleep(2)
await device.abort()
try:
    await running
except AbortedError as error:
    print("stopped:", error)



## Run a protocol file

A `.LHC` protocol file is read into a `Protocol`: what it will run, and everything the file records
alongside it. Reading needs `pycryptodome`, installed at the top of this notebook.

Reading a file never fails on a step it cannot understand — the file's own records are kept as they
are, so a protocol from another model still reads, prints and writes. `build_steps()` is what turns
those records into steps, and it names the one that will not read.

In [ ]:
from pylabrobot.agilent.biotek.lhc import read

protocol = read("/home/stefan/workspace/biotek_plr/pylabrobot_modified/pylabrobot/agilent/biotek/lhc/tests/test_data/protocols/405_TS_and_LS/001_W-CORNING_FLAT_96.LHC")

print("name:      ", protocol.protocol_name)
print("written by:", protocol.lhc_version)
print("written for:", protocol.instrument_name)
print("plate:     ", protocol.plate_type or protocol.plate_type_number)
print("entries:   ", len(protocol.entries), "of which", len(protocol.device_entries), "operate the instrument")

for index, step in enumerate(protocol.build_steps()):
    print(f"  step {index}: {type(step).__name__}")

### What the file says about its instrument

A protocol file records the options the instrument had fitted when it was written. Nothing runs
against that record — steps are encoded against the instrument in front of you — and nothing writes
it to the instrument. It is good for exactly one question, worth asking about a file that came from
another machine: was this written for a differently equipped washer?

`compare_settings()` is truthy when the two agree, and prints as the options that differ. It raises
`ValueError` for a file that carries no such record, which is how the oldest releases wrote one.

In [ ]:
comparison = device.compare_settings(protocol)
print(comparison)
print("same configuration:", bool(comparison))

### Running it

`run_protocol()` checks the protocol, opens one batch around the whole run, sends each step and
polls it to completion. The check is the same `can_run()` from above and happens automatically, so
a protocol that cannot run raises before anything moves.

**This runs whatever the protocol does**, which for most washer protocols means filling and
emptying every well of the plate on the carrier. Read the steps printed above first.

The entries that sequence a run rather than operate the instrument — delays, loops, remarks — are
not run; the device steps go in file order. They are still there on `protocol.entries` to
inspect.

In [ ]:
await device.run_protocol(protocol, home_on_close=True)

Steps built in Python run the same way. `run_protocol()` takes a list of steps as readily as a
protocol, and `run_step()` runs a single one.

In [ ]:
await device.run_protocol(
    [
        ManifoldPrime(volume=10_000, buffer="A", flow_rate=9),
        ManifoldPrime(volume=5_000, buffer="B", flow_rate=9),
    ],
    home_on_close=True,
)

## Home the transport and disconnect

Homing drives the transport to its home position and confirms it arrived. Do it before a person
reaches for the plate, unless the last batch already closed with `home_on_close=True`.

`stop()` closes the link. It does nothing on an instrument that is already closed, so it is safe to
run this cell twice, and it is worth running from a `finally` in a script so that a failed run does
not leave the port open.

In [ ]:
# await device.home()
# await device.stop()

# validation mode: close the mirror after the device, so the last operations are checked and
# journalled before the report is read. Closing the pool shuts down the wine processes.
await device.home()
await device.stop()

report = mirror.close()
pool.close()
journal.close()
print(report)

```{note}
The fluid left in the manifold after a run is the instrument's problem, not the driver's. Follow the
manufacturer's shutdown and maintenance procedure — `device.washer.auto_clean(...)` soaks the
manifold in cleaning fluid, and most maintenance routines ship as protocol files you can run with
`run_protocol()`.
```

## Validation mode: read the log back

`shadow.jsonl` holds every frame both sides sent and every reply each was given, one JSON object
per line. `view.render` lays a run out per operation, this driver's frames on the left and the
vendor DLL's on the right.

The two markers to look for:

- `<<<` — the position where the two sides sent different frames.
- `SYN` — the emulator had no recorded reply for a frame the DLL sent, so it made one up. That
  means the DLL asked something this driver never asked, and it is usually the reason for every
  difference after it.

`only_different=True` skips the operations that agreed, which is how a long run is read.

In [ ]:
from plr_shadow import view

records = view.load(JOURNAL_PATH)
print(view.render(records))

One operation on its own, whatever its verdict — useful once the summary
above has named the one worth looking at:

In [ ]:
print(view.render(records, operation=1))

The records are plain dictionaries, so anything the rendering does not show is a
comprehension away. Note that they are not in run order: the checked operations are journalled
from the mirror's worker thread and the skipped ones from the thread the device runs on, so sort
by `operation` to recover the sequence. Every frame this driver put on the wire:

In [ ]:
frames = [
    record
    for record in records
    if record["event"] == "frame" and record["side"] == "plr" and record["direction"] == "sent"
]

for record in sorted(frames, key=lambda one: (one["operation"], one["index"])):
    print(record["operation"], record["command_name"], record["payload"])